# Notebook 2.4i — System-Wide ESC Simulation (Compiled)

Refactored and compiled version of notebook 2.4h.

**Model:** `0209_clusteredSE` (Negative Binomial Regression with clustered SE)
**Scope:** All BoSY system learners x total contracted ESC slots
**Goal:** Quantify the ESC system's role in decongesting public JHS under multiple subsidy scenarios

**Improvements over 2.4h:**
- Scenario sweep refactored into a single loop (no copy-paste)
- Decongestion computation extracted into reusable functions
- Results table built programmatically (not manually transcribed)
- `congested_frac` approximation validated against simulation data
- Over-enrollment flagged separately from model under-prediction
- Consistent billed totals throughout
- Baseline-relative percent changes added alongside sequential

# 0. Setup

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from collections import OrderedDict

pd.set_option('display.max_columns', None)

PROJECT_DIR = Path.cwd().parent
OUTPUT_DIR = PROJECT_DIR / 'output'

print(f'Project directory: {PROJECT_DIR}')

Project directory: /workspace/project_paaral


# 1. Load & Prepare Data

In [2]:
# --- Load raw data ---

# School metadata
sch_info = pd.read_parquet(OUTPUT_DIR / 'processed_project_bukas_school_information.parquet')
sch_info_slim = sch_info[['school_id', 'school_name', 'old_region', 'division', 'sector']].copy()
sch_info_slim['school_id'] = sch_info_slim['school_id'].astype(str)

# Observed student flow (G7 SY2324)
student_flow = pd.read_parquet(OUTPUT_DIR / 'grade_7_student_flow_table_sy2324.parquet')

# ESC slots
esc_slots = pd.read_parquet(OUTPUT_DIR / 'processed_esc_slots.parquet')
esc_slots['school_id'] = esc_slots['school_id'].astype(str)

# Predicted flow from NBR
cbp = pd.read_parquet(
    OUTPUT_DIR / 'full_candidate_beneficiary_pool_without_probdist_0209_clusteredSE.parquet'
)
cbp['origin_school_id'] = cbp['origin_school_id'].astype(str)
cbp['destination_school_id'] = cbp['destination_school_id'].astype(str)

# Tag existing vs hypothetical paths
existing_keys = set(
    student_flow['school_id_origin'] + '_' + student_flow['school_id_destination']
)
cbp['is_hypothetical'] = ~(
    cbp['origin_school_id'] + '_' + cbp['destination_school_id']
).isin(existing_keys)

# Origins that feed congested public JHS
flow_to_congested = pd.read_parquet(
    OUTPUT_DIR / 'analysis_payload' / 'flow_to_congested.parquet'
)
flow_to_congested['school_id_origin'] = flow_to_congested['school_id_origin'].astype(str)
congested_feeding_origins = set(flow_to_congested['school_id_origin'].unique())

print(f"School info:            {sch_info.shape[0]:>10,} schools")
print(f"Student flow:           {student_flow.shape[0]:>10,} rows")
print(f"ESC slots (raw):        {esc_slots.shape[0]:>10,} rows")
print(f"CBP:                    {cbp.shape[0]:>10,} pairs")
print(f"  Existing paths:       {(~cbp['is_hypothetical']).sum():>10,}")
print(f"  Hypothetical paths:   {cbp['is_hypothetical'].sum():>10,}")
print(f"Congested-feeding origins: {len(congested_feeding_origins):>7,}")

School info:                61,442 schools
Student flow:              288,328 rows
ESC slots (raw):             4,638 rows
CBP:                       325,395 pairs
  Existing paths:           90,423
  Hypothetical paths:      234,972
Congested-feeding origins:   8,015


In [3]:
# --- Build system_slots ---
# Filter ESC slots to schools that appear as CBP destinations only
# (ESC schools are destinations, not origins — clean filter)

cbp_dest_ids = set(cbp['destination_school_id'].unique())

# Per our team's discussion, we will use reported ESC beneficiaries instead of slots_billed
agg_student_flow_dest = student_flow.groupby(['school_id_destination'], as_index=False)['count_esc_beneficiary'].sum()
esc_slots_obs = esc_slots.merge(
    agg_student_flow_dest,
    left_on='school_id',
    right_on='school_id_destination',
    how='left'
).drop(columns=['school_id_destination'])

system_slots = (
    esc_slots_obs[esc_slots_obs['school_id'].isin(cbp_dest_ids)]
    .drop(columns=['esc_school_id', 'school_name', 'has_deped_school_id'], errors='ignore')
    .groupby('school_id', as_index=False)
    .agg({'slots_total': 'sum', 'slots_billed': 'sum', 'slots_unutilized': 'sum', 'count_esc_beneficiary': 'sum'})
)

# Lookups
benef_lookup = system_slots.set_index('school_id')['count_esc_beneficiary'].to_dict()
billed_lookup = system_slots.set_index('school_id')['slots_billed'].to_dict()
slots_total_lookup = system_slots.set_index('school_id')['slots_total'].to_dict()

n_over = (system_slots['slots_unutilized'] < 0).sum()
over_excess = -system_slots.loc[system_slots['slots_unutilized'] < 0, 'slots_unutilized'].sum()

print(f"ESC schools in system:  {len(system_slots):>7,}")
print(f"  CBP unique destinations: {len(cbp_dest_ids):>4,}")
print(f"  Slots total:               {system_slots['slots_total'].sum():>10,}")
print(f"  ESC beneficiaries (flow):  {system_slots['count_esc_beneficiary'].sum():>10,.0f}")
print(f"  Slots billed (admin):      {system_slots['slots_billed'].sum():>10,}")
print(f"  Utilization (flow):        {system_slots['count_esc_beneficiary'].sum() / system_slots['slots_total'].sum() * 100:>9.2f}%")
print(f"  Utilization (admin):       {system_slots['slots_billed'].sum() / system_slots['slots_total'].sum() * 100:>9.2f}%")
print(f"  Over-enrolled schools: {n_over:>6,} ({over_excess:,} excess students)")

ESC schools in system:    1,373
  CBP unique destinations: 1,373
  Slots total:                  106,654
  ESC beneficiaries (flow):      76,438
  Slots billed (admin):          82,071
  Utilization (flow):            71.67%
  Utilization (admin):           76.95%
  Over-enrolled schools:    188 (1,748 excess students)


In [4]:
# --- Tag observed flows with destination type ---
# For each origin-destination pair in the student flow table, flag whether the
# destination is a congested public JHS or an ESC school.

congested_dest_ids = set(flow_to_congested['school_id_destination'].unique())

df_flow_tagged = student_flow.copy()
df_flow_tagged['school_id_origin'] = df_flow_tagged['school_id_origin'].astype(str)
df_flow_tagged['school_id_destination'] = df_flow_tagged['school_id_destination'].astype(str)

df_flow_tagged['is_congested_destination'] = (
    df_flow_tagged['school_id_destination'].isin(congested_dest_ids)
)
df_flow_tagged['is_esc_destination'] = (
    df_flow_tagged['school_id_destination'].isin(cbp_dest_ids)
)

# Total students per flow = non-beneficiary + ESC beneficiary
df_flow_tagged['total_students'] = (
    df_flow_tagged['count_non_beneficiary'].fillna(0)
    + df_flow_tagged['count_esc_beneficiary'].fillna(0)
)

n_total = df_flow_tagged['total_students'].sum()
n_to_congested = df_flow_tagged.loc[
    df_flow_tagged['is_congested_destination'], 'total_students'
].sum()
n_to_esc_all = df_flow_tagged.loc[
    df_flow_tagged['is_esc_destination'], 'total_students'
].sum()
n_to_esc_benef = df_flow_tagged.loc[
    df_flow_tagged['is_esc_destination'], 'count_esc_beneficiary'
].fillna(0).sum()

print(f"Tagged flows: {len(df_flow_tagged):,} origin-destination pairs")
print(f"  Total students:                    {n_total:>10,.0f}")
print(f"  To congested public JHS:           {n_to_congested:>10,.0f} ({n_to_congested/n_total*100:.1f}%)")
print(f"  All students at ESC schools:       {n_to_esc_all:>10,.0f} ({n_to_esc_all/n_total*100:.1f}%)")
print(f"    of which ESC beneficiaries:      {n_to_esc_benef:>10,.0f}")
print(f"    of which non-beneficiaries:      {n_to_esc_all - n_to_esc_benef:>10,.0f}")
print(f"  To other destinations:             {n_total - n_to_congested - n_to_esc_all:>10,.0f}")
print(f"Congested public JHS:                {len(congested_dest_ids):>7,}")
print(f"ESC schools (CBP system):            {len(cbp_dest_ids):>7,}")

Tagged flows: 288,328 origin-destination pairs
  Total students:                     1,788,916
  To congested public JHS:              416,458 (23.3%)
  All students at ESC schools:           88,663 (5.0%)
    of which ESC beneficiaries:          74,232
    of which non-beneficiaries:          14,431
  To other destinations:              1,283,795
Congested public JHS:                  1,254
ESC schools (CBP system):              1,373


In [5]:
# --- Build candidate pool ---
# Aggregate observed G7 flows by origin, filtered to CBP origins only
# (candidate pool = all students from origins in the system, including ESC beneficiaries)

cbp_origin_ids = set(cbp['origin_school_id'].unique())

flow_by_origin = (
    student_flow
    .groupby('school_id_origin', as_index=False)
    .agg(
        count_non_beneficiary=('count_non_beneficiary', 'sum'),
        count_esc_beneficiary=('count_esc_beneficiary', 'sum'),
    )
)
flow_by_origin['school_id_origin'] = flow_by_origin['school_id_origin'].astype(str)

# Filter to CBP origins only
flow_by_origin = flow_by_origin[
    flow_by_origin['school_id_origin'].isin(cbp_origin_ids)
].copy()

flow_by_origin['candidate_pool'] = (
    flow_by_origin['count_non_beneficiary'].fillna(0)
    + flow_by_origin['count_esc_beneficiary'].fillna(0)
)

total_candidates = flow_by_origin['candidate_pool'].sum()
total_esc_obs = flow_by_origin['count_esc_beneficiary'].sum()

print(f"Origin schools:                      {len(flow_by_origin):>7,}")
print(f"Candidate pool:                      {total_candidates:>10,.0f}")
print(f"  ESC beneficiaries (flow, CBP origins): {total_esc_obs:>6,.0f}")

Origin schools:                        7,622
Candidate pool:                         562,958
  ESC beneficiaries (flow, CBP origins): 73,624


In [25]:
display(flow_by_origin.head(3))
display(flow_by_origin['count_esc_beneficiary'].sum())

,school_id_origin,count_non_beneficiary,count_esc_beneficiary,candidate_pool
1607,101701,41.0,0.0,41.0
2075,102178,45.0,0.0,45.0
2153,102259,33.0,1.0,34.0


np.float64(73624.0)

In [6]:
# --- System overview ---
total_slots = system_slots['slots_total'].sum()
total_esc_benef = system_slots['count_esc_beneficiary'].sum()
total_billed = system_slots['slots_billed'].sum()

# Unutilized vs over-enrolled breakdown
mask_surplus = system_slots['slots_unutilized'] > 0
mask_over = system_slots['slots_unutilized'] < 0
slots_unutilized = system_slots.loc[mask_surplus, 'slots_unutilized'].sum()
slots_over_excess = -system_slots.loc[mask_over, 'slots_unutilized'].sum()

# Mu vs billed: how well does the model predict current enrollment?
mu_per_dest = cbp.groupby('destination_school_id')['mu_current_subsidy'].sum()
billed_series = system_slots.set_index('school_id')['slots_billed']
mu_vs_billed = (mu_per_dest - billed_series).dropna()

print("=" * 60)
print("PRE-SIMULATION SYSTEM OVERVIEW")
print("=" * 60)

print(f"\n--- Demand Side ---")
print(f"Origin schools:                          {len(flow_by_origin):>10,}")
print(f"  Congested-feeding origins:             {len(congested_feeding_origins):>10,}")
print(f"Candidate pool (all BoSY learners):      {total_candidates:>10,.0f}")
print(f"  ESC beneficiaries (flow, CBP origins): {total_esc_obs:>10,.0f}")

print(f"\n--- Supply Side ---")
print(f"ESC schools in system:                   {len(system_slots):>10,}")
print(f"Total contracted slots:                  {total_slots:>10,}")
print(f"ESC beneficiaries (flow):                {total_esc_benef:>10,.0f}")
print(f"Slots billed (admin):                    {total_billed:>10,}")
print(f"All students at ESC schools:             {n_to_esc_all:>10,.0f}")
print(f"Utilization (flow):                      {total_esc_benef / total_slots * 100:>9.1f}%")
print(f"Utilization (admin):                     {total_billed / total_slots * 100:>9.1f}%")
print(f"Unutilized slots ({mask_surplus.sum()} schools):          {slots_unutilized:>10,.0f}")
print(f"Over-enrolled excess ({mask_over.sum()} schools):        {slots_over_excess:>10,.0f}")

print(f"\n--- Model (CBP) ---")
print(f"Origin-destination pairs:                {len(cbp):>10,}")
print(f"  Existing paths:                        {(~cbp['is_hypothetical']).sum():>10,}")
print(f"  Hypothetical paths:                    {cbp['is_hypothetical'].sum():>10,}")
print(f"Mean(predicted mu - billed):             {mu_vs_billed.mean():>+10,.1f} students/school")
print(f"Std(predicted mu - billed):              {mu_vs_billed.std():>10,.1f} students/school")

print(f"\n--- Constraints ---")
print(f"Demand/supply ratio:                     {total_candidates / total_slots:>9.1f}x")
print(f"Congested public JHS:                    {len(congested_dest_ids):>10,}")

PRE-SIMULATION SYSTEM OVERVIEW

--- Demand Side ---
Origin schools:                               7,622
  Congested-feeding origins:                  8,015
Candidate pool (all BoSY learners):         562,958
  ESC beneficiaries (flow, CBP origins):     73,624

--- Supply Side ---
ESC schools in system:                        1,373
Total contracted slots:                     106,654
ESC beneficiaries (flow):                    76,438
Slots billed (admin):                        82,071
All students at ESC schools:                 88,663
Utilization (flow):                           71.7%
Utilization (admin):                          77.0%
Unutilized slots (1072 schools):              26,331
Over-enrolled excess (188 schools):             1,748

--- Model (CBP) ---
Origin-destination pairs:                   325,395
  Existing paths:                            90,423
  Hypothetical paths:                       234,972
Mean(predicted mu - billed):                 +550.7 students/school
Std

# 2. Observed Baseline

Before running any simulations, examine where students actually go.
This provides a ground truth to compare against model predictions later.

Two levels of analysis:
- **Origin-level:** For each origin school, what fraction of its students flow to congested public JHS vs ESC schools?
- **ESC-school-level:** For each ESC school, what fraction of its actual enrollees come from origins that also feed congested public JHS?

In [7]:
# --- 2.1 Origin-level: Where do students actually go? ---

# Per-origin totals
total_by_origin = df_flow_tagged.groupby('school_id_origin')['total_students'].sum()

# Per-origin flow to congested destinations
to_congested_by_origin = (
    df_flow_tagged[df_flow_tagged['is_congested_destination']]
    .groupby('school_id_origin')['total_students'].sum()
)

# Per-origin flow to ESC destinations
to_esc_by_origin = (
    df_flow_tagged[df_flow_tagged['is_esc_destination']]
    .groupby('school_id_origin')['total_students'].sum()
)

obs_by_origin = pd.DataFrame({
    'total_students': total_by_origin,
    'students_to_congested': to_congested_by_origin,
    'students_to_esc': to_esc_by_origin,
}).fillna(0)

obs_by_origin['pct_to_congested'] = (
    obs_by_origin['students_to_congested'] / obs_by_origin['total_students'] * 100
)
obs_by_origin['pct_to_esc'] = (
    obs_by_origin['students_to_esc'] / obs_by_origin['total_students'] * 100
)
obs_by_origin['is_congested_feeding'] = obs_by_origin.index.isin(congested_feeding_origins)

# Summary
n_origins = len(obs_by_origin)
n_cf = obs_by_origin['is_congested_feeding'].sum()
n_any_congested = (obs_by_origin['students_to_congested'] > 0).sum()
n_any_esc = (obs_by_origin['students_to_esc'] > 0).sum()

print(f"Origins in student flow: {n_origins:,}")
print(f"  Tagged as congested-feeding: {n_cf:,}")
print(f"  With any flow to congested:  {n_any_congested:,}")
print(f"  With any flow to ESC:        {n_any_esc:,}")
print()

# Compare congested-feeding vs non-congested-feeding origins
cf_mask = obs_by_origin['is_congested_feeding']
print("Mean observed flow distribution:")
print(f"  Congested-feeding origins ({cf_mask.sum():,}):")
print(f"    To congested: {obs_by_origin.loc[cf_mask, 'pct_to_congested'].mean():.1f}%")
print(f"    To ESC:       {obs_by_origin.loc[cf_mask, 'pct_to_esc'].mean():.1f}%")
print(f"  Non-congested-feeding origins ({(~cf_mask).sum():,}):")
print(f"    To congested: {obs_by_origin.loc[~cf_mask, 'pct_to_congested'].mean():.1f}%")
print(f"    To ESC:       {obs_by_origin.loc[~cf_mask, 'pct_to_esc'].mean():.1f}%")

Origins in student flow: 47,179
  Tagged as congested-feeding: 8,015
  With any flow to congested:  15,082
  With any flow to ESC:        10,709

Mean observed flow distribution:
  Congested-feeding origins (8,015):
    To congested: 55.8%
    To ESC:       20.1%
  Non-congested-feeding origins (39,164):
    To congested: 1.2%
    To ESC:       4.1%


In [8]:
# --- 2.2 ESC-school-level: Observed congested fraction ---
# For each ESC school, what fraction of its actual enrollees come from
# origins that also feed congested public JHS?
# This is the observed analogue of the model's congested_frac.

esc_flows = df_flow_tagged[df_flow_tagged['is_esc_destination']].copy()

# Total observed students per ESC school (all students, incl. non-beneficiaries)
obs_esc_total = esc_flows.groupby('school_id_destination')['total_students'].sum()

# ESC beneficiaries per ESC school
obs_esc_benef = esc_flows.groupby('school_id_destination')['count_esc_beneficiary'].sum()

# Students from congested-feeding origins per ESC school (all students)
obs_esc_from_cf = (
    esc_flows[esc_flows['school_id_origin'].isin(congested_feeding_origins)]
    .groupby('school_id_destination')['total_students'].sum()
)

# ESC beneficiaries from congested-feeding origins per ESC school
obs_esc_benef_from_cf = (
    esc_flows[esc_flows['school_id_origin'].isin(congested_feeding_origins)]
    .groupby('school_id_destination')['count_esc_beneficiary'].sum()
)

obs_by_esc = pd.DataFrame({
    'obs_total_students': obs_esc_total,
    'obs_esc_beneficiaries': obs_esc_benef,
    'obs_from_congested_origins': obs_esc_from_cf,
    'obs_benef_from_congested': obs_esc_benef_from_cf,
}).fillna(0)

obs_by_esc['obs_congested_frac'] = (
    obs_by_esc['obs_from_congested_origins'] / obs_by_esc['obs_total_students']
).fillna(0)

n_esc_observed = len(obs_by_esc)
mean_obs_frac = obs_by_esc.loc[obs_by_esc['obs_congested_frac'] > 0, 'obs_congested_frac'].mean()

print(f"ESC schools with observed student flow:   {n_esc_observed:,}")
print(f"  All students at ESC schools:            {obs_by_esc['obs_total_students'].sum():>10,.0f}")
print(f"  ESC beneficiaries (flow):               {obs_by_esc['obs_esc_beneficiaries'].sum():>10,.0f}")
print(f"  All students from congested origins:    {obs_by_esc['obs_from_congested_origins'].sum():>10,.0f}")
print(f"  ESC beneficiaries from congested origins:{obs_by_esc['obs_benef_from_congested'].sum():>9,.0f}")
print(f"  Mean observed congested fraction:       {mean_obs_frac:.1%}")
print()
print("Distribution of observed congested fraction:")
print(obs_by_esc['obs_congested_frac'].describe().to_string())

ESC schools with observed student flow:   1,369
  All students at ESC schools:                88,663
  ESC beneficiaries (flow):                   74,232
  All students from congested origins:        66,509
  ESC beneficiaries from congested origins:   55,188
  Mean observed congested fraction:       73.3%

Distribution of observed congested fraction:
count    1369.000000
mean        0.727739
std         0.200706
min         0.000000
25%         0.633663
50%         0.777778
75%         0.870690
max         1.000000


# 3. Pre-compute Decongestion Weights

For each ESC school, compute `congested_frac`: the share of its NBR-predicted demand
that comes from origins also feeding congested public JHS.

This is computed from raw (unconstrained) mu values as a proxy.
We validate this approximation against simulation results in Section 6.

In [9]:
# --- congested_frac: per-ESC-school congested-feeding fraction ---
# Denominator: total predicted demand per ESC school (all origins)
mu_per_dest_all = cbp.groupby('destination_school_id')['mu_current_subsidy'].sum()

# Numerator: predicted demand from congested-feeding origins only
mu_per_dest_congested = (
    cbp[cbp['origin_school_id'].isin(congested_feeding_origins)]
    .groupby('destination_school_id')['mu_current_subsidy'].sum()
)

congested_frac = (mu_per_dest_congested / mu_per_dest_all).fillna(0).to_dict()

n_with = sum(1 for v in congested_frac.values() if v > 0)
mean_frac = np.mean([v for v in congested_frac.values() if v > 0])

print(f"ESC schools with congested-feeding demand: {n_with:,} / {len(congested_frac):,}")
print(f"Mean congested-feeding fraction:           {mean_frac:.1%}")
print(f"Raw system-level congested fraction:       {mu_per_dest_congested.sum() / mu_per_dest_all.sum():.4f}")

ESC schools with congested-feeding demand: 1,370 / 1,373
Mean congested-feeding fraction:           94.4%
Raw system-level congested fraction:       0.9879


# 4. Simulation

In [10]:
# --- Build simulation pools and pre-extract arrays ---

# Candidate pool: {origin_id: candidate_count}
CAND_POOL = flow_by_origin.set_index('school_id_origin')['candidate_pool'].to_dict()

# Slot pool: {dest_id: slots_total}
# Only schools with predicted demand (all system_slots entries, since filtered to CBP destinations)
SLOT_POOL = system_slots.set_index('school_id')['slots_total'].to_dict()

# Pre-extract static arrays (reused across all scenarios and iterations)
ARR_ORIGIN = cbp['origin_school_id'].values
ARR_DEST = cbp['destination_school_id'].values
ARR_HYPO = cbp['is_hypothetical'].values
N_PATHS = len(cbp)

print(f"CAND_POOL: {len(CAND_POOL):,} origins, {sum(CAND_POOL.values()):,.0f} total candidates")
print(f"SLOT_POOL: {len(SLOT_POOL):,} destinations, {sum(SLOT_POOL.values()):,.0f} total slots")
print(f"N_PATHS:   {N_PATHS:,}")

CAND_POOL: 7,622 origins, 562,958 total candidates
SLOT_POOL: 1,373 destinations, 106,654 total slots
N_PATHS:   325,395


In [11]:
def run_simulation_detailed(mu_values, cand_pool_init, slot_pool_init, seed):
    """
    Run one iteration of the constrained path simulation with school-level tracking.

    Coupled depletion: each accepted diversion reduces both the origin's candidate pool
    and the destination's slot pool. Paths are shuffled randomly each iteration.

    Returns dict with:
        - System-level: total_existing, total_hypothetical, total_both
        - Per-school: per_dest {dest_id: [exist, hypo]}, per_origin {origin_id: [exist, hypo]}
    """
    tracker_cand = dict(cand_pool_init)
    tracker_slots = dict(slot_pool_init)
    remaining_cand = sum(cand_pool_init.values())
    remaining_slots = sum(slot_pool_init.values())

    rng = np.random.default_rng(seed)
    indices = np.arange(N_PATHS)
    rng.shuffle(indices)

    total_existing = 0.0
    total_hypothetical = 0.0
    per_dest = {}
    per_origin = {}

    for idx in indices:
        if remaining_cand <= 0 or remaining_slots <= 0:
            break

        mu_val = mu_values[idx]
        if mu_val <= 0 or np.isnan(mu_val):
            continue

        og_id = ARR_ORIGIN[idx]
        dest_id = ARR_DEST[idx]

        current_pool = tracker_cand.get(og_id, 0)
        current_slots = tracker_slots.get(dest_id, 0)

        if current_pool > 0 and current_slots > 0:
            accepted = min(current_pool, current_slots, mu_val)
            tracker_cand[og_id] -= accepted
            tracker_slots[dest_id] -= accepted
            remaining_cand -= accepted
            remaining_slots -= accepted

            slot = 1 if ARR_HYPO[idx] else 0  # 0=existing, 1=hypothetical

            if ARR_HYPO[idx]:
                total_hypothetical += accepted
            else:
                total_existing += accepted

            if dest_id not in per_dest:
                per_dest[dest_id] = [0.0, 0.0]
            per_dest[dest_id][slot] += accepted

            if og_id not in per_origin:
                per_origin[og_id] = [0.0, 0.0]
            per_origin[og_id][slot] += accepted

    return {
        'total_existing': total_existing,
        'total_hypothetical': total_hypothetical,
        'total_both': total_existing + total_hypothetical,
        'remaining_candidates': remaining_cand,
        'remaining_slots': remaining_slots,
        'per_dest': per_dest,
        'per_origin': per_origin,
    }

print(f"Simulation function defined. {N_PATHS:,} paths per iteration.")

Simulation function defined. 325,395 paths per iteration.


In [12]:
def compute_decongestion_stats(detailed_results, congested_frac, billed_lookup,
                                congested_feeding_origins):
    """
    Compute system-level decongestion statistics from detailed MC results.

    Two approaches:
      Question A (marginal): max(0, predicted - billed) x congested_frac
      Question B (total predicted): predicted x congested_frac

    Also computes actual_from_congested_origins using per_origin data
    for validating the congested_frac approximation.

    Returns DataFrame with one row per iteration.
    """
    iteration_stats = []

    for i, r in enumerate(detailed_results):
        per_dest = r['per_dest']
        per_origin = r['per_origin']

        total_predicted = 0.0
        total_marginal = 0.0
        marginal_from_congested = 0.0
        marginal_existing = 0.0
        marginal_hypothetical = 0.0
        predicted_congested = 0.0
        predicted_congested_existing = 0.0
        predicted_congested_hypothetical = 0.0
        schools_with_marginal = 0

        for dest_id, (exist, hypo) in per_dest.items():
            predicted = exist + hypo
            total_predicted += predicted
            billed = billed_lookup.get(dest_id, 0)
            frac = congested_frac.get(dest_id, 0)

            # Question B: total predicted congested-feeding flow
            predicted_congested += predicted * frac
            predicted_congested_existing += exist * frac
            predicted_congested_hypothetical += hypo * frac

            # Question A: marginal (only positive)
            marginal = max(0, predicted - billed)
            if marginal > 0:
                schools_with_marginal += 1
                total_marginal += marginal
                marginal_from_congested += marginal * frac
                if predicted > 0:
                    marginal_existing += marginal * (exist / predicted)
                    marginal_hypothetical += marginal * (hypo / predicted)

        # Actual system-level congested fraction from per_origin
        from_congested_origins = sum(
            vals[0] + vals[1]
            for og_id, vals in per_origin.items()
            if og_id in congested_feeding_origins
        )

        iteration_stats.append({
            'iteration': i,
            'total_predicted': total_predicted,
            'predicted_congested': predicted_congested,
            'predicted_congested_existing': predicted_congested_existing,
            'predicted_congested_hypothetical': predicted_congested_hypothetical,
            'total_marginal': total_marginal,
            'marginal_from_congested': marginal_from_congested,
            'marginal_existing': marginal_existing,
            'marginal_hypothetical': marginal_hypothetical,
            'schools_with_marginal': schools_with_marginal,
            'actual_from_congested_origins': from_congested_origins,
        })

    return pd.DataFrame(iteration_stats)

print("compute_decongestion_stats() defined.")

compute_decongestion_stats() defined.


In [13]:
def build_school_dataframe(detailed_results, congested_frac, billed_lookup,
                            slots_total_lookup, sch_info_slim):
    """
    Build per-ESC-school DataFrame with decongestion metrics from MC results.

    Adds is_over_enrolled flag to distinguish structural negative marginals
    (billed > slots_total) from genuine model under-predictions.
    """
    all_dest_ids = set()
    for r in detailed_results:
        all_dest_ids.update(r['per_dest'].keys())

    school_rows = []
    for dest_id in all_dest_ids:
        exist_vals, hypo_vals = [], []
        for r in detailed_results:
            vals = r['per_dest'].get(dest_id, [0.0, 0.0])
            exist_vals.append(vals[0])
            hypo_vals.append(vals[1])

        mean_exist = np.mean(exist_vals)
        mean_hypo = np.mean(hypo_vals)
        mean_predicted = mean_exist + mean_hypo
        billed = billed_lookup.get(dest_id, 0)
        slots_total = slots_total_lookup.get(dest_id, 0)
        frac = congested_frac.get(dest_id, 0)

        raw_marginal = mean_predicted - billed
        pos_marginal = max(0, raw_marginal)

        school_rows.append({
            'destination_school_id': dest_id,
            'slots_total': slots_total,
            'slots_billed': billed,
            'is_over_enrolled': billed > slots_total,
            'mean_predicted': mean_predicted,
            'mean_existing': mean_exist,
            'mean_hypothetical': mean_hypo,
            'pct_hypothetical': (mean_hypo / mean_predicted * 100) if mean_predicted > 0 else 0,
            'raw_marginal': raw_marginal,
            'congested_frac': frac,
            'predicted_congested': mean_predicted * frac,
            'marginal_decongestion': pos_marginal * frac,
            'marginal_existing': (
                pos_marginal * (mean_exist / mean_predicted)
                if mean_predicted > 0 and pos_marginal > 0 else 0
            ),
            'marginal_hypothetical': (
                pos_marginal * (mean_hypo / mean_predicted)
                if mean_predicted > 0 and pos_marginal > 0 else 0
            ),
        })

    df = pd.DataFrame(school_rows)
    df = df.merge(
        sch_info_slim[['school_id', 'school_name', 'division']],
        left_on='destination_school_id',
        right_on='school_id',
        how='left',
    ).drop(columns=['school_id'])

    return df.sort_values('marginal_decongestion', ascending=False).reset_index(drop=True)

print("build_school_dataframe() defined.")

build_school_dataframe() defined.


# 5. Run Scenarios

Run the constrained path simulation for multiple subsidy scenarios.
Each scenario uses 100 MC iterations with coupled depletion.

In [14]:
%%time
SCENARIOS = OrderedDict([
    # ('current_subsidy', 'mu_current_subsidy'),
    ('-1k_net_cost',    'mu_minus_1k_net_cost'),
    ('-5k_net_cost',    'mu_minus_5k_net_cost'),
    ('-10k_net_cost',   'mu_minus_10k_net_cost'),
    ('-15k_net_cost',   'mu_minus_15k_net_cost'),
    ('-20k_net_cost',   'mu_minus_20k_net_cost'),
])

N_ITERATIONS = 100 # 200

all_detailed_results = {}
all_decongestion_stats = {}

for label, mu_col in SCENARIOS.items():
    mu_arr = cbp[mu_col].values
    results = []
    for i in tqdm(range(N_ITERATIONS), desc=label):
        result = run_simulation_detailed(
            mu_values=mu_arr,
            cand_pool_init=CAND_POOL,
            slot_pool_init=SLOT_POOL,
            seed=i,
        )
        results.append(result)

    all_detailed_results[label] = results
    all_decongestion_stats[label] = compute_decongestion_stats(
        results, congested_frac, billed_lookup, congested_feeding_origins
    )

    mean_both = np.mean([r['total_both'] for r in results])
    print(f"  {label}: mean_predicted = {mean_both:,.1f}")

print(f"\nCompleted {len(SCENARIOS)} scenarios x {N_ITERATIONS} iterations.")

-1k_net_cost: 100%|██████████| 100/100 [00:35<00:00,  2.84it/s]


  -1k_net_cost: mean_predicted = 99,991.7


-5k_net_cost: 100%|██████████| 100/100 [00:34<00:00,  2.94it/s]


  -5k_net_cost: mean_predicted = 100,441.5


-10k_net_cost: 100%|██████████| 100/100 [00:33<00:00,  2.97it/s]


  -10k_net_cost: mean_predicted = 100,944.0


-15k_net_cost: 100%|██████████| 100/100 [00:33<00:00,  2.98it/s]


  -15k_net_cost: mean_predicted = 101,427.7


-20k_net_cost: 100%|██████████| 100/100 [00:33<00:00,  2.98it/s]


  -20k_net_cost: mean_predicted = 101,818.0

Completed 5 scenarios x 100 iterations.
CPU times: user 2min 50s, sys: 1.15 s, total: 2min 51s
Wall time: 2min 51s


# 6. Results

In [15]:
# --- Validate congested_frac approximation ---
# Compare raw-mu-based fraction vs actual simulation-based fraction at system level
# Use the first (lowest subsidy) scenario for validation

raw_system_frac = mu_per_dest_congested.sum() / mu_per_dest_all.sum()

first_scenario = list(SCENARIOS.keys())[0]
stats_first = all_decongestion_stats[first_scenario]
actual_system_frac = (
    stats_first['actual_from_congested_origins'].mean()
    / stats_first['total_predicted'].mean()
)

print(f"System-level congested fraction validation (scenario: {first_scenario}):")
print(f"  From raw mu (pre-computed approximation): {raw_system_frac:.4f}")
print(f"  From simulation (per_origin, MC mean):    {actual_system_frac:.4f}")
print(f"  Absolute difference:                      {abs(raw_system_frac - actual_system_frac):.4f}")
diff_pct = abs(raw_system_frac - actual_system_frac) / raw_system_frac * 100
print(f"  Relative difference:                      {diff_pct:.2f}%")
print()
if diff_pct < 1.0:
    print("  Raw mu approximation is valid at system level (< 1% relative difference).")
    print("  Per-school fractions are proxies — exact values would require pair-level tracking.")
else:
    print("  WARNING: Notable difference. Per-school fractions may need recalibration.")

System-level congested fraction validation (scenario: -1k_net_cost):
  From raw mu (pre-computed approximation): 0.9879
  From simulation (per_origin, MC mean):    0.9569
  Absolute difference:                      0.0310
  Relative difference:                      3.14%



In [16]:
# --- Cross-scenario comparison table ---

def ci_str(series):
    return f"[{series.quantile(0.025):,.1f}, {series.quantile(0.975):,.1f}]"

# Observed baseline row: ESC beneficiaries only (not all students at ESC schools)
# This is the correct comparison for the simulation, which predicts ESC enrollment
obs_flow = obs_by_esc['obs_esc_beneficiaries'].sum()
obs_congested = obs_by_esc['obs_benef_from_congested'].sum()

rows = [{
    'scenario': 'observed (benef)',
    'mean_pred_flow': obs_flow,
    'mean_pred_congested': obs_congested,
    'ci_pred_congested': '—',
    'mean_marginal': np.nan,
    'mean_marginal_congested': np.nan,
    'ci_marginal_congested': '—',
    'hypo_share_pct': np.nan,
    'schools_w_marginal': np.nan,
}]

for label in SCENARIOS:
    s = all_decongestion_stats[label]
    rows.append({
        'scenario': label,
        'mean_pred_flow': s['total_predicted'].mean(),
        'mean_pred_congested': s['predicted_congested'].mean(),
        'ci_pred_congested': ci_str(s['predicted_congested']),
        'mean_marginal': s['total_marginal'].mean(),
        'mean_marginal_congested': s['marginal_from_congested'].mean(),
        'ci_marginal_congested': ci_str(s['marginal_from_congested']),
        'hypo_share_pct': s['marginal_hypothetical'].mean() / s['total_marginal'].mean() * 100,
        'schools_w_marginal': s['schools_with_marginal'].mean(),
    })

df_results = pd.DataFrame(rows).set_index('scenario')

# Percent change from observed baseline
df_results['pct_from_observed_flow'] = (
    (df_results['mean_pred_flow'] - obs_flow) / obs_flow * 100
)
df_results['pct_from_observed_congested'] = (
    (df_results['mean_pred_congested'] - obs_congested) / obs_congested * 100
)

# # Percent change from current_subsidy (for scenario comparison)
# baseline = df_results.loc['-1k_net_cost']
# df_results['pct_from_baseline_flow'] = (
#     (df_results['mean_pred_flow'] - baseline['mean_pred_flow'])
#     / baseline['mean_pred_flow'] * 100
# )

display(
    df_results.style.format({
        'mean_pred_flow': '{:,.1f}',
        'mean_pred_congested': '{:,.1f}',
        'mean_marginal': '{:,.1f}',
        'mean_marginal_congested': '{:,.1f}',
        'hypo_share_pct': '{:.1f}%',
        'schools_w_marginal': '{:,.0f}',
        'pct_from_observed_flow': '{:+.2f}%',
        'pct_from_observed_congested': '{:+.2f}%',
        # 'pct_from_baseline_flow': '{:+.3f}%',
    }, na_rep='—')
)

,mean_pred_flow,mean_pred_congested,ci_pred_congested,mean_marginal,mean_marginal_congested,ci_marginal_congested,hypo_share_pct,schools_w_marginal,pct_from_observed_flow,pct_from_observed_congested
scenario,,,,,,,,,,
observed (benef),"74,232.0","55,188.0",—,—,—,—,—,—,+0.00%,+0.00%
-1k_net_cost,"99,991.7","95,695.1","[95,639.9, 95,740.2]","25,293.4","24,820.2","[24,782.4, 24,858.9]",53.9%,"1,065",+34.70%,+73.40%
-5k_net_cost,"100,441.5","96,096.9","[96,048.0, 96,136.2]","25,425.6","24,948.0","[24,909.7, 24,990.2]",53.6%,"1,070",+35.31%,+74.13%
-10k_net_cost,"100,944.0","96,536.1","[96,482.4, 96,573.8]","25,536.6","25,055.8","[25,021.1, 25,093.7]",53.5%,"1,070",+35.98%,+74.92%
-15k_net_cost,"101,427.7","96,942.2","[96,899.0, 96,981.7]","25,587.8","25,104.6","[25,072.5, 25,139.5]",53.4%,"1,071",+36.64%,+75.66%
-20k_net_cost,"101,818.0","97,268.0","[97,223.8, 97,306.1]","25,631.3","25,146.3","[25,116.2, 25,178.5]",53.3%,"1,072",+37.16%,+76.25%


In [19]:
fmt = "{:,.0f}"
ess_cols = [
    'mean_pred_flow',
    'mean_pred_congested',
    'hypo_share_pct',
    'pct_from_observed_flow',
    'pct_from_observed_congested'
]
display(
    df_results[ess_cols]
    .style.format(
        {
            'mean_pred_flow':fmt,
            'mean_pred_congested':fmt,
            'hypo_share_pct':'{:+.2f}%',
            'pct_from_observed_flow':'{:+.2f}%',
            'pct_from_observed_congested':'{:+.2f}%',
        }
    )
)

,mean_pred_flow,mean_pred_congested,hypo_share_pct,pct_from_observed_flow,pct_from_observed_congested
scenario,,,,,
observed (benef),"74,232","55,188",+nan%,+0.00%,+0.00%
-1k_net_cost,"99,992","95,695",+53.85%,+34.70%,+73.40%
-5k_net_cost,"100,442","96,097",+53.64%,+35.31%,+74.13%
-10k_net_cost,"100,944","96,536",+53.48%,+35.98%,+74.92%
-15k_net_cost,"101,428","96,942",+53.39%,+36.64%,+75.66%
-20k_net_cost,"101,818","97,268",+53.32%,+37.16%,+76.25%


In [26]:
# --- Detailed statistics for lowest-subsidy scenario (-1k_net_cost) ---

s = all_decongestion_stats['-1k_net_cost']
total_billed = system_slots['slots_billed'].sum()

print("=" * 65)
print("-1k NET COST -- Detailed Decongestion Statistics")
print("=" * 65)
print(f"\nTotal billed ({len(system_slots)} ESC schools): {total_billed:,}")
print(f"Mean predicted flow:                    {s['total_predicted'].mean():>10,.1f}")

print(f"\n--- Question B: Total Predicted Decongestion Role ---")
print(f"Mean predicted congested flow:          {s['predicted_congested'].mean():>10,.1f}")
print(f"  95% CI: {ci_str(s['predicted_congested'])}")
print(f"  Existing:                             {s['predicted_congested_existing'].mean():>10,.1f}")
print(f"  Hypothetical:                         {s['predicted_congested_hypothetical'].mean():>10,.1f}")
pct_b = s['predicted_congested_hypothetical'].mean() / s['predicted_congested'].mean() * 100
print(f"  Hypothetical share:                   {pct_b:>9.1f}%")

print(f"\n--- Question A: Marginal Decongestion ---")
print(f"Mean total marginal:                    {s['total_marginal'].mean():>10,.1f}")
print(f"  95% CI: {ci_str(s['total_marginal'])}")
print(f"Mean marginal from congested:           {s['marginal_from_congested'].mean():>10,.1f}")
print(f"  95% CI: {ci_str(s['marginal_from_congested'])}")
pct_c = s['marginal_from_congested'].mean() / s['total_marginal'].mean() * 100
print(f"  = {pct_c:.1f}% of total marginal")
print(f"  Existing:                             {s['marginal_existing'].mean():>10,.1f}")
print(f"  Hypothetical:                         {s['marginal_hypothetical'].mean():>10,.1f}")
pct_a = s['marginal_hypothetical'].mean() / s['total_marginal'].mean() * 100
print(f"  Hypothetical share:                   {pct_a:>9.1f}%")

print(f"\nSchools with positive marginal:         {s['schools_with_marginal'].mean():>10,.0f}")

-1k NET COST -- Detailed Decongestion Statistics

Total billed (1373 ESC schools): 82,071
Mean predicted flow:                      99,991.7

--- Question B: Total Predicted Decongestion Role ---
Mean predicted congested flow:            95,695.1
  95% CI: [95,639.9, 95,740.2]
  Existing:                               54,497.2
  Hypothetical:                           41,197.9
  Hypothetical share:                        43.1%

--- Question A: Marginal Decongestion ---
Mean total marginal:                      25,293.4
  95% CI: [25,255.4, 25,333.3]
Mean marginal from congested:             24,820.2
  95% CI: [24,782.4, 24,858.9]
  = 98.1% of total marginal
  Existing:                               11,672.3
  Hypothetical:                           13,621.1
  Hypothetical share:                        53.9%

Schools with positive marginal:              1,065


In [27]:
# --- Per-ESC-school DataFrame for -1k_net_cost scenario ---

df_school = build_school_dataframe(
    all_detailed_results['-1k_net_cost'],
    congested_frac, billed_lookup, slots_total_lookup, sch_info_slim,
)

n_positive = (df_school['raw_marginal'] > 0).sum()
n_negative = (df_school['raw_marginal'] < 0).sum()
n_neg_over = ((df_school['raw_marginal'] < 0) & df_school['is_over_enrolled']).sum()
n_neg_under = ((df_school['raw_marginal'] < 0) & ~df_school['is_over_enrolled']).sum()

print(f"ESC schools in simulation: {len(df_school)}")
print(f"  Positive marginal (predicted > billed): {n_positive}")
print(f"  Negative marginal (predicted < billed): {n_negative}")
print(f"    -- over-enrolled (billed > slots_total): {n_neg_over}")
print(f"    -- model under-prediction:               {n_neg_under}")
print(f"  Zero marginal: {len(df_school) - n_positive - n_negative}")
print()
print(f"Total predicted congested flow:    {df_school['predicted_congested'].sum():>10,.1f}")
print(f"Total marginal decongestion:       {df_school['marginal_decongestion'].sum():>10,.1f}")

ESC schools in simulation: 1370
  Positive marginal (predicted > billed): 1057
  Negative marginal (predicted < billed): 284
    -- over-enrolled (billed > slots_total): 185
    -- model under-prediction:               99
  Zero marginal: 29

Total predicted congested flow:      95,695.1
Total marginal decongestion:         24,818.3


In [28]:
# --- Inspection ---
fmt = {
    'slots_total': '{:,.0f}',
    'slots_billed': '{:,.0f}',
    'mean_predicted': '{:,.1f}',
    'mean_existing': '{:,.1f}',
    'mean_hypothetical': '{:,.1f}',
    'pct_hypothetical': '{:.0f}%',
    'raw_marginal': '{:+,.1f}',
    'congested_frac': '{:.0%}',
    'predicted_congested': '{:,.1f}',
    'marginal_decongestion': '{:,.1f}',
    'marginal_existing': '{:,.1f}',
    'marginal_hypothetical': '{:,.1f}',
}

# Top 10 by marginal decongestion
print("=== Top 10 ESC Schools by Marginal Decongestion ===")
display(df_school.head(10).style.format(fmt))

# --- Other slices (uncomment as needed) ---

# Schools where model under-predicts (excluding over-enrollment)
# display(
#     df_school[(df_school['raw_marginal'] < 0) & ~df_school['is_over_enrolled']]
#     .sort_values('raw_marginal').head(20).style.format(fmt)
# )

# Top by total predicted decongestion role
# display(df_school.sort_values('predicted_congested', ascending=False).head(20).style.format(fmt))

# By division
# display(df_school.groupby('division').agg(
#     n_schools=('destination_school_id', 'count'),
#     total_marginal=('marginal_decongestion', 'sum'),
#     total_predicted_congested=('predicted_congested', 'sum'),
#     mean_congested_frac=('congested_frac', 'mean'),
# ).sort_values('total_marginal', ascending=False))

# Over-enrolled schools
# display(df_school[df_school['is_over_enrolled']].sort_values('raw_marginal').head(20).style.format(fmt))

=== Top 10 ESC Schools by Marginal Decongestion ===


,destination_school_id,slots_total,slots_billed,is_over_enrolled,mean_predicted,mean_existing,mean_hypothetical,pct_hypothetical,raw_marginal,congested_frac,predicted_congested,marginal_decongestion,marginal_existing,marginal_hypothetical,school_name,division
0,401984,325,169,False,325.0,222.5,102.5,32%,+156.0,99%,320.4,153.8,106.8,49.2,Kin Yang Academy Inc.,Dasmarinas City
1,401439,338,204,False,338.0,312.8,25.2,7%,+134.0,98%,330.2,130.9,124.0,10.0,"La Concepcion College, Inc.",City of San Jose Del Monte
2,407079,218,87,False,218.0,105.7,112.3,52%,+131.0,100%,217.0,130.4,63.5,67.5,"Bernardo College (Santiago G. Bernardo Foundation, Inc.)",Las Piñas City
3,402501,280,149,False,280.0,199.6,80.4,29%,+131.0,97%,271.0,126.8,93.4,37.6,St. Vincent College of Cabuyao,Cabuyao City
4,400839,200,79,False,200.0,82.0,118.0,59%,+121.0,98%,196.8,119.0,49.6,71.4,"MEYCAUAYAN COLLEGE, INC.",Meycauayan City
5,487518,460,341,False,460.0,362.5,97.5,21%,+119.0,100%,458.5,118.6,93.8,25.2,Jose Rizal High School - Arellano University,Malabon City
6,401121,334,215,False,334.0,297.3,36.7,11%,+119.0,96%,319.3,113.8,105.9,13.1,St. Augustine Academy of Pampanga,Pampanga
7,401317,495,377,False,495.0,432.8,62.2,13%,+118.0,95%,469.0,111.8,103.2,14.8,Holy Angel University,Angeles City
8,401698,325,212,False,325.0,268.3,56.7,17%,+113.0,97%,314.4,109.3,93.3,19.7,Tanauan Institute Inc.,Tanauan City
9,401640,420,308,False,420.0,390.7,29.3,7%,+112.0,97%,408.5,108.9,104.2,7.8,Santo Niño Formation & Science School,Batangas


## 6.4. Observed vs Predicted: Origin-Level

Compare the observed flow distribution (Section 2) with the model's predictions.

For each origin, the model predicts how many students would go to ESC schools (via mu).
The observed data shows how many actually went. We also check whether origins tagged
as "congested-feeding" truly send a large share of their students to congested public JHS.

In [29]:
# --- Origin-level: Observed vs Predicted ESC enrollment ---

# Model: predicted total ESC enrollment per origin (sum of mu across all ESC destinations)
model_esc_by_origin = (
    cbp.groupby('origin_school_id')['mu_current_subsidy'].sum()
    .rename('model_predicted_esc')
)

# Join observed and predicted
compare_origin = obs_by_origin.join(model_esc_by_origin, how='outer').fillna(0)

# Filter to origins that appear in both observed and model
in_both = (compare_origin['total_students'] > 0) & (compare_origin['model_predicted_esc'] > 0)
compare_active = compare_origin[in_both].copy()

# Observed ESC students per origin
# (use count_esc_beneficiary where available, fall back to total ESC-destination flow)
obs_esc_per_origin = (
    df_flow_tagged[df_flow_tagged['is_esc_destination']]
    .groupby('school_id_origin')['total_students'].sum()
    .rename('obs_esc_students')
)
compare_active = compare_active.join(obs_esc_per_origin, how='left').fillna(0)

print(f"Origins in both observed and model: {len(compare_active):,}")
print()

# Validate congested-feeding tag
cf = compare_active[compare_active['is_congested_feeding']]
ncf = compare_active[~compare_active['is_congested_feeding']]

print("Validation of 'congested-feeding' origin tag:")
print(f"  Congested-feeding origins ({len(cf):,}):")
print(f"    Mean observed pct to congested: {cf['pct_to_congested'].mean():.1f}%")
print(f"    Mean observed pct to ESC:       {cf['pct_to_esc'].mean():.1f}%")
print(f"    Mean model predicted ESC:       {cf['model_predicted_esc'].mean():.1f}")
print(f"    Mean observed ESC students:     {cf['obs_esc_students'].mean():.1f}")
print(f"  Non-congested-feeding origins ({len(ncf):,}):")
print(f"    Mean observed pct to congested: {ncf['pct_to_congested'].mean():.1f}%")
print(f"    Mean observed pct to ESC:       {ncf['pct_to_esc'].mean():.1f}%")
print(f"    Mean model predicted ESC:       {ncf['model_predicted_esc'].mean():.1f}")
print(f"    Mean observed ESC students:     {ncf['obs_esc_students'].mean():.1f}")
print()

# Correlation between observed and predicted ESC enrollment per origin
corr = compare_active[['obs_esc_students', 'model_predicted_esc']].corr().iloc[0, 1]
print(f"Correlation (observed vs predicted ESC per origin): {corr:.3f}")

Origins in both observed and model: 7,622

Validation of 'congested-feeding' origin tag:
  Congested-feeding origins (6,554):
    Mean observed pct to congested: 53.0%
    Mean observed pct to ESC:       24.1%
    Mean model predicted ESC:       126.3
    Mean observed ESC students:     10.1
  Non-congested-feeding origins (1,068):
    Mean observed pct to congested: 0.0%
    Mean observed pct to ESC:       67.3%
    Mean model predicted ESC:       9.5
    Mean observed ESC students:     10.6

Correlation (observed vs predicted ESC per origin): 0.325


## 6.5. Observed vs Predicted: ESC-School-Level

For each ESC school, compare the model's `congested_frac` (share of predicted beneficiary demand
from congested-feeding origins, computed from raw mu) with the observed congested fraction.

Two observed versions:
- **All-student**: `obs_from_congested_origins / obs_total_students` — includes non-beneficiaries
- **Beneficiary-based**: `obs_benef_from_congested / obs_esc_beneficiaries` — apples-to-apples with the model, since mu predicts beneficiary flow

The beneficiary-based comparison is the correct validation of the decongestion weighting.

In [30]:
# --- ESC-school-level: Observed vs Model congested fraction ---

# Model's congested_frac (from raw mu, computed in Section 3)
model_cf = pd.Series(congested_frac, name='model_congested_frac')

# Join with observed (from Section 2)
compare_esc = obs_by_esc.join(model_cf, how='inner')

# Beneficiary-based observed congested fraction (apples-to-apples with model)
# Model's congested_frac is from mu ratios, and mu predicts beneficiary flow
compare_esc['obs_benef_congested_frac'] = (
    compare_esc['obs_benef_from_congested'] / compare_esc['obs_esc_beneficiaries']
).fillna(0)

# Also join slots info for context
compare_esc = compare_esc.join(
    system_slots.set_index('school_id')[['slots_total', 'slots_billed']],
    how='left',
)

# Join school names
compare_esc = compare_esc.join(
    sch_info_slim.set_index('school_id')[['school_name', 'division']],
    how='left',
)

print(f"ESC schools with both observed and model data: {len(compare_esc):,}")

# --- System-level comparison ---
print(f"\n{'='*65}")
print("System-level congested fraction comparison")
print(f"{'='*65}")

# All-student observed
obs_sys_all = (
    compare_esc['obs_from_congested_origins'].sum()
    / compare_esc['obs_total_students'].sum()
)
# Beneficiary-based observed
obs_sys_benef = (
    compare_esc['obs_benef_from_congested'].sum()
    / compare_esc['obs_esc_beneficiaries'].sum()
)
# Model (weighted by observed enrollment for comparability)
model_sys = (
    (compare_esc['model_congested_frac'] * compare_esc['obs_total_students']).sum()
    / compare_esc['obs_total_students'].sum()
)

print(f"  Observed (all students):       {obs_sys_all:.4f}")
print(f"  Observed (beneficiaries only): {obs_sys_benef:.4f}")
print(f"  Model (from mu ratios):        {model_sys:.4f}")
print(f"  Diff (model - obs all):        {model_sys - obs_sys_all:+.4f}")
print(f"  Diff (model - obs benef):      {model_sys - obs_sys_benef:+.4f}")

# --- Per-school correlation ---
print(f"\n{'='*65}")
print("Per-school correlation with model congested_frac")
print(f"{'='*65}")

corr_all = compare_esc[['obs_congested_frac', 'model_congested_frac']].corr().iloc[0, 1]
corr_benef = compare_esc[['obs_benef_congested_frac', 'model_congested_frac']].corr().iloc[0, 1]
print(f"  vs observed (all students):    {corr_all:.3f}")
print(f"  vs observed (beneficiaries):   {corr_benef:.3f}")

# --- Distribution of differences (beneficiary-based) ---
compare_esc['frac_diff_all'] = compare_esc['model_congested_frac'] - compare_esc['obs_congested_frac']
compare_esc['frac_diff_benef'] = compare_esc['model_congested_frac'] - compare_esc['obs_benef_congested_frac']

print(f"\n{'='*65}")
print("Distribution of (model - observed) congested fraction")
print(f"{'='*65}")
print("\nAll-student basis:")
print(compare_esc['frac_diff_all'].describe().to_string())
print("\nBeneficiary basis:")
print(compare_esc['frac_diff_benef'].describe().to_string())

# --- Top discrepancies (beneficiary-based) ---
print()
print("=== Top 10 schools: model overestimates (beneficiary basis) ===")
display(
    compare_esc.nlargest(10, 'frac_diff_benef')[
        ['school_name', 'obs_esc_beneficiaries', 'obs_benef_congested_frac',
         'model_congested_frac', 'frac_diff_benef']
    ].style.format({
        'obs_esc_beneficiaries': '{:,.0f}',
        'obs_benef_congested_frac': '{:.2%}',
        'model_congested_frac': '{:.2%}',
        'frac_diff_benef': '{:+.2%}',
    })
)

ESC schools with both observed and model data: 1,369

System-level congested fraction comparison
  Observed (all students):       0.7501
  Observed (beneficiaries only): 0.7435
  Model (from mu ratios):        0.9361
  Diff (model - obs all):        +0.1859
  Diff (model - obs benef):      +0.1926

Per-school correlation with model congested_frac
  vs observed (all students):    0.345
  vs observed (beneficiaries):   0.344

Distribution of (model - observed) congested fraction

All-student basis:
count    1369.000000
mean        0.214150
std         0.193623
min        -0.306011
25%         0.082679
50%         0.169243
75%         0.294294
max         0.996482

Beneficiary basis:
count    1369.000000
mean        0.220841
std         0.193781
min        -0.264755
25%         0.087463
50%         0.179204
75%         0.302589
max         1.000000

=== Top 10 schools: model overestimates (beneficiary basis) ===


,school_name,obs_esc_beneficiaries,obs_benef_congested_frac,model_congested_frac,frac_diff_benef
402945,Victory Elijah Christian College,0,0.00%,100.00%,+100.00%
424409,Oxford Philippines Int'l. School,0,0.00%,100.00%,+100.00%
482045,"St. Lino Science High School, Inc.",0,0.00%,100.00%,+100.00%
486038,UPSouth Education Foundation Inc.,3,0.00%,99.65%,+99.65%
406956,"Aquinas School, Inc.",12,0.00%,99.53%,+99.53%
403038,"Queen Mary Learning Center of Cainta, Inc.",3,0.00%,99.50%,+99.50%
406859,"M.A. Montessori School, Inc.",2,0.00%,98.88%,+98.88%
403008,COLEGIO STO. DOMINGO EDUCATIONAL FOUNDATION INC.,11,0.00%,98.10%,+98.10%
401788,Marella Christianne Institute,3,0.00%,95.76%,+95.76%
424605,"Saint Bridgette Integrated School, Inc.",3,0.00%,95.61%,+95.61%


# 7. Key Findings

**Slot-constrained, not price-constrained:**

- Across 5 subsidy scenarios (-1k to -20k net cost reduction), predicted flow ranges from 99,992 to 101,818 — a mere **+1.8%** increase even at the maximum subsidy reduction
- The binding constraint is the 106,654 total contracted slots, not the price of ESC. Demand exceeds supply 5.3x
- Policy implication: **slot expansion and geographic reallocation** (see notebook 2.4g) should be prioritized over subsidy increases

**Decongestion structure (from -1k scenario, representative of all scenarios):**

- **Question B** — Total predicted congested flow: **95,695** out of 99,992 total predicted (**95.7%** of all predicted ESC enrollment comes from congested-feeding origins)
  - Existing paths: 54,497 | Hypothetical paths: 41,198 (43.1% hypothetical)
- **Question A** — Marginal decongestion (predicted - billed, clamped at 0): **25,293** total marginal, of which **24,820 from congested-feeding origins** (98.1%)
  - Existing: 11,672 | Hypothetical: 13,621 (**53.9% from hypothetical paths** — untapped routes)
  - 1,057 schools with positive marginal; 284 with negative (185 over-enrolled, 99 model under-prediction)

**Observed vs predicted:**

- Observed ESC beneficiaries (flow): **74,232** vs model-predicted ~100,000 under slot constraint. The gap reflects that the simulation uses `slots_total` (full contracted capacity), not just currently billed slots
- Congested-feeding origin tag validated: tagged origins send 53.0% of students to congested public JHS vs 0.0% for non-tagged
- Per-ESC-school `congested_frac`: model systematically overestimates (mean model 0.94 vs observed 0.74, per-school correlation 0.34). The model's raw-mu-based congested fraction is a coarser proxy than the observed beneficiary-based fraction
- `congested_frac` approximation: 3.1% relative difference at system level (raw mu 0.988 vs simulation-actual 0.957) — within acceptable range for system-level analysis but per-school fractions are proxies

## 7.1. Interpretation

**The capacity gap reveals untapped decongestion potential.** The ESC system currently enrolls 74,232 beneficiaries, but the model predicts ~100,000 students would fill slots if all 106,654 contracted slots were fully utilized. The ~25,000 marginal students represent the additional absorption capacity beyond current enrollment, and 98% of them would come from origins that also feed congested public JHS. In other words, virtually all of the ESC system's unused capacity is geographically positioned to relieve congested schools. However, 1,072 ESC schools currently have 26,331 unutilized slots — the barrier is not a lack of demand (demand exceeds supply 5.3x) but that students have not yet been matched to these available slots.

**Subsidy reduction is not the lever; the slot ceiling is.** Reducing net cost by 20,000 pesos — effectively making ESC free — only adds ~1,826 students (+1.8%) to the predicted flow. This is because distance is the dominant factor in the NBR model (approximately 4x more important than cost). When 563,000 candidates compete for 107,000 slots, cheaper tuition shuffles which students fill slots but barely changes the total. The policy implication is that expanding the number of contracted ESC slots, or reallocating existing slots from surplus to deficit areas (see notebook 2.4g), would be far more effective than increasing the subsidy.

**Over half the untapped decongestion potential lies in currently unobserved paths.** Of the ~25,000 marginal students, 53.9% (13,621) would flow through origin-destination pairs where no student currently enrolls. These hypothetical paths are geographically feasible (predicted by the NBR model based on distance, cost, and school characteristics) but have not yet been activated. This suggests that information barriers or inertia — not just capacity — may limit the ESC system's decongestion reach. Targeted outreach to students from congested-feeding origins about nearby ESC schools they have not yet considered could activate these latent pathways.

**The ESC system's decongestion role is structurally significant but bounded.** Currently, 416,458 students flow to the 1,254 congested public JHS (23.3% of all G7 students). The model's predicted congested-feeding flow of ~95,695 represents about 23% of that congestion volume — a substantial structural counterweight. The marginal component (~24,820 additional students) represents about 6% of the congested flow. Even full utilization of existing ESC capacity would not eliminate congestion, but it would meaningfully reduce pressure on the most crowded public schools.

**The model's congested fraction is directionally correct but overstated at the school level.** The model assigns 94% of predicted demand to congested-feeding origins, whereas the observed beneficiary-based fraction is 74%. This ~20 percentage point gap arises because the NBR's distance-dominated predictions concentrate predicted flow on nearby origins — which tend to be congested-feeding — while actual enrollment is more geographically diffuse. At the system level, this means the 95.7% "congested share" of predicted flow is an upper bound; the true decongestion-relevant share is likely closer to the observed 74%. The core finding — that the large majority of ESC flow is decongestion-relevant — holds under either estimate.